In [1]:
!pip install -q langchain==0.1.16 langchain-openai==0.0.8 langchain-community==0.0.32

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requir

In [2]:
!pip install --upgrade numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 85.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.0.32 requires numpy<2,>=1, but you have numpy 2.4.2 which is incompatible.
langchain 0.1.16 requires numpy<2,>=1, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
db-dtypes 1.5.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.


In [3]:
from langchain_openai import ChatOpenAI
import os
import pandas as pd

from langchain.agents import AgentType, initialize_agent,load_tools
from langchain.tools import Tool


In [ ]:
# # local model
# llm = ChatOpenAI(
#     openai_api_base="http://localhost:1234/v1",
#     openai_api_key="lm_studio",
#     model="llama-3.2-1b-instruct",
#     temperature=0.9
# )

In [ ]:
llm = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key="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",
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0.5
)

os.environ["SERPAPI_API_KEY"] = "23acdbd8baf763c985ad5eea7b45d863f1d82ee1144368c79f848e1acdf6b91e"


In [ ]:
!git clone https://github.com/deepanrajm/deep_learning.git

In [ ]:
EXCEL_PATH = "/content/deep_learning/agents/expense.xlsx"

In [ ]:
def analyze_expenses_excel(query: str) -> str:
    """
    Analyze salary and expenses from Excel using pandas.
    """
    df = pd.read_excel(EXCEL_PATH)

    salary = df.loc[df["Category"].str.lower() == "salary", "Monthly_Amount"].sum()

    expenses_df = df[df["Category"].str.lower() != "salary"]

    total_expenses = expenses_df["Monthly_Amount"].sum()
    savings = salary - total_expenses
    savings_percentage = (savings / salary) * 100 if salary > 0 else 0

    top_expenses = (
        expenses_df
        .sort_values(by="Monthly_Amount", ascending=False)
        .head(3)[["Category", "Monthly_Amount"]]
        .to_dict(orient="records")
    )

    summary = {
        "monthly_salary": salary,
        "total_monthly_expenses": total_expenses,
        "monthly_savings": savings,
        "savings_percentage": round(savings_percentage, 2),
        "top_3_expense_categories": top_expenses
    }

    return str(summary)


In [ ]:
expense_tool = Tool(
    name="expense_analyzer",
    description=(
        "Analyze salary and monthly expenses from Excel and return "
        "savings, spending breakdown, and high expense categories."
    ),
    func=analyze_expenses_excel
)


In [ ]:
pip install google-search-results

In [ ]:
tools = load_tools(
    ["serpapi", "llm-math"],
    llm=llm
)

tools.append(expense_tool)


In [ ]:
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)


In [ ]:
query = """
This is my monthly salary and expense sheet.
Please analyze it and tell me:
1. How much I am saving
2. Where I am overspending
3. How I can reduce expenses
4. How to improve my savings
"""

result = agent.run(query)
print(result)
